# langchain

## RecursiveCharacterTextSplitter

In [7]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text = """
LangChain is a framework for developing applications powered by language models.

It enables applications that are:
- Data-aware
- Agentic
"""

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20
)

chunks = text_splitter.split_text(text)

for i, chunk in enumerate(chunks):
    print(f"Chunk {i}:\n{chunk}\n---")


Chunk 0:
LangChain is a framework for developing applications powered by language models.
---
Chunk 1:
It enables applications that are:
- Data-aware
- Agentic
---


## MarkdownHeaderTextSplitter

In [8]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

markdown_text = """
# LangChain

LangChain 是一个用于构建 LLM 应用的框架。

## Text Splitters

用于将文档切分成小块。

### RecursiveCharacterTextSplitter

这是最常用的切分器。
"""

headers_to_split_on = [
    ("#", "h1"),
    ("##", "h2"),
    ("###", "h3"),
]

splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)

docs = splitter.split_text(markdown_text)

for doc in docs:
    print(doc.metadata)
    print(doc.page_content)
    print("----")


{'h1': 'LangChain'}
LangChain 是一个用于构建 LLM 应用的框架。
----
{'h1': 'LangChain', 'h2': 'Text Splitters'}
用于将文档切分成小块。
----
{'h1': 'LangChain', 'h2': 'Text Splitters', 'h3': 'RecursiveCharacterTextSplitter'}
这是最常用的切分器。
----


## TokenTextSplitter

In [9]:
from langchain_text_splitters import TokenTextSplitter

text = "LangChain helps developers build applications powered by LLMs."

splitter = TokenTextSplitter(
    chunk_size=10,
    chunk_overlap=2
)

chunks = splitter.split_text(text)

for c in chunks:
    print(c)


LangChain helps developers build applications powered by LL
 by LLMs.


In [10]:
from langchain_text_splitters import (
    MarkdownHeaderTextSplitter,
    TokenTextSplitter
)


markdown_text = """
# LangChain

LangChain 是一个用于构建 LLM 应用的框架。

## Text Splitters

用于将文档切分成小块。

### RecursiveCharacterTextSplitter

这是最常用的切分器。
"""

docs = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#", "h1"),
        ("##", "h2"),
        ("###", "h3"),
    ]
).split_text(markdown_text)

final_docs = TokenTextSplitter(
    chunk_size=512,
    chunk_overlap=64
).split_documents(docs)


for doc in docs:
    print(doc.metadata)
    print(doc.page_content)
    print("----")


{'h1': 'LangChain'}
LangChain 是一个用于构建 LLM 应用的框架。
----
{'h1': 'LangChain', 'h2': 'Text Splitters'}
用于将文档切分成小块。
----
{'h1': 'LangChain', 'h2': 'Text Splitters', 'h3': 'RecursiveCharacterTextSplitter'}
这是最常用的切分器。
----


In [11]:
import tiktoken # type: ignore
from langchain_text_splitters import TokenTextSplitter

enc = tiktoken.encoding_for_model("gpt-4o")
splitter = TokenTextSplitter(chunk_size=100, chunk_overlap=10)

print(len(enc.encode("LangChain helps developers build applications powered by LLMs.")))


12


# 量化和蒸馏区别

| 维度             | **量化 (Quantization)** | **蒸馏 (Distillation)** |
| ---------------- | ----------------------- | ----------------------- |
| **本质**         | 降低数值精度            | 训练小模型模仿大模型    |
| **参数数量**     | 不变（仍是 7B）         | 减少（如 7B → 1.3B）    |
| **模型结构**     | 不变                    | 改变（层数/宽度减少）   |
| **是否需要训练** | ❌ 否（加载时转换）      | ✅ 是（需大量数据+GPU）  |
| **可逆性**       | ✅ 理论可恢复            | ❌ 不可逆                |

# milvus

## 使用本地 Embedding 模型生成向量


In [3]:
import time
from langchain_huggingface import HuggingFaceEmbeddings


texts = ["这是一个测试文本"] * 10
# 修改这一行
model_path = "D:/ollama/bge-base-zh-v1.5"

embeddings = HuggingFaceEmbeddings(
    model_name=model_path,
    model_kwargs={"device": "cpu"},
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": 16
    }
)

start = time.time()
vectors = embeddings.embed_documents(texts)
print("耗时:", time.time() - start)
dim = len(vectors[0])
print("向量维度:", dim)


耗时: 0.25255465507507324
向量维度: 768




```text
文本
 └─> Embedding（768维 float32）
      └─> Milvus Collection
           ├─ id (INT64, PK)
           ├─ vector (FLOAT_VECTOR, dim=768)
           └─ text (VARCHAR)
```

## 连接 milvus

In [4]:
import os
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from pymilvus import MilvusClient, DataType # 👈 只需要导入这两个

# 1. 尝试从环境变量获取 Token
# 第二个参数是默认值，如果没找到环境变量会报错或使用空字符串，建议做个判断
token = os.getenv("ZILLIZ_TOKEN")

if not token:
    raise ValueError("❌ 未找到环境变量 'ZILLIZ_TOKEN'，请检查配置！")

print(f"🔌 正在连接 Zilliz Cloud...")
# ✅ 修复点 1: 建立连接 (直接实例化 Client，不需要 connections.connect)
ZILLIZ_URI="https://in03-f410f8aefc6d1ee.serverless.gcp-us-west1.cloud.zilliz.com"
client = MilvusClient(uri=ZILLIZ_URI, token=token)
print("connected")

🔌 正在连接 Zilliz Cloud...
connected


## 创建表

In [5]:
# ===========================
# 3. 数据入库 (Create & Insert)
# ===========================
COLLECTION_NAME = "demo_text_search"

# ✅ 修复点 2: 删除旧表 (Client 直接调用，不需要 utility)
if client.has_collection(COLLECTION_NAME):
    client.drop_collection(COLLECTION_NAME)

# ✅ 修复点 3: 创建表结构 (Create Schema)
# 新版 API 定义 Schema 更直观
schema = client.create_schema(
    auto_id=True, 
    enable_dynamic_field=False, 
    description="测试新版API"
)
schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)
schema.add_field(field_name="text", datatype=DataType.VARCHAR, max_length=1000)
schema.add_field(field_name="vector", datatype=DataType.FLOAT_VECTOR, dim=dim)

# ✅ 修复点 4: 创建索引参数 (Index Params)
index_params = client.prepare_index_params()
index_params.add_index(
    field_name="vector", 
    metric_type="COSINE", 
    index_type="AUTOINDEX"
)

# 正式建表 (一步到位：建表 + 建索引)
client.create_collection(
    collection_name=COLLECTION_NAME,
    schema=schema,
    index_params=index_params
)
print(f"📦 集合 {COLLECTION_NAME} 创建完成")

📦 集合 demo_text_search 创建完成


## 插入数据

In [6]:
# ==========================================
# 4. 数据整理与插入 (Insert)
# ==========================================
# ✅ 修复点 5: 数据格式转换
# MilvusClient 推荐使用 "列表嵌套字典" 的格式 (Rows)，而不是原来的按列存储

documents = [
    "Milvus 是一个云原生的开源向量数据库。",
    "Zilliz Cloud 提供了全托管服务。",
    "BGE 是一个强大的中文向量模型。",
    "Python 是一种广泛用于 AI 的语言。",
    "今天天气真不错。"
]

data_rows = []
for doc, vec in zip(documents, vectors):
    data_rows.append({
        "text": doc,
        "vector": vec
    })

res = client.insert(collection_name=COLLECTION_NAME, data=data_rows)
print(f"📥 成功插入 {res['insert_count']} 条数据")

📥 成功插入 5 条数据


## 建立索引并加载

In [7]:
# ==========================================
# 5. 搜索 (Search)
# ==========================================
query_text = "什么数据库是云原生的？"
print(f"\n🔎 提问: {query_text}")
query_vector = embeddings.embed_query(query_text)

# ✅ 修复点 6: 执行搜索 (不再返回 Future/Coroutine，直接返回 list)
results = client.search(
    collection_name=COLLECTION_NAME,
    data=[query_vector],
    limit=2,
    output_fields=["text"], # 必填，否则只返回 ID
    search_params={"metric_type": "COSINE", "params": {}}
)

for hits in results:
    for hit in hits:
        print("-" * 30)
        # client.search 返回的结果直接就是字典，不需要 .entity.get()
        print(f"匹配内容: {hit['entity']['text']}")
        print(f"相似度:   {hit['distance']:.4f}")


🔎 提问: 什么数据库是云原生的？
------------------------------
匹配内容: Milvus 是一个云原生的开源向量数据库。
相似度:   0.3282
------------------------------
匹配内容: Zilliz Cloud 提供了全托管服务。
相似度:   0.3282


## 完整示例 MilvusClient 

[cloud.zilliz.com](https://cloud.zilliz.com/orgs/org-vlekjlebfuesfhnyqekngc/projects/proj-247c1825702aee5dafd1b2/clusters/in03-f410f8aefc6d1ee)

In [8]:
import os
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from pymilvus import MilvusClient, DataType # 👈 只需要导入这两个

# ==========================================
# 1. 初始化 (加载环境变量)
# ==========================================
load_dotenv()
ZILLIZ_URI = "https://in03-f410f8aefc6d1ee.serverless.gcp-us-west1.cloud.zilliz.com"
ZILLIZ_TOKEN = os.getenv("ZILLIZ_TOKEN")

if not ZILLIZ_TOKEN:
    print("❌ 错误：未找到 ZILLIZ_TOKEN，请检查 .env 文件！")
    exit()

# ==========================================
# 2. 准备模型与数据 (保持不变)
# ==========================================
model_path = "D:/ollama/bge-base-zh-v1.5"
print(f"🧠 正在加载模型: {model_path} ...")
embeddings = HuggingFaceEmbeddings(
    model_name=model_path,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 16}
)

documents = [
    "Milvus 是一个云原生的开源向量数据库。",
    "Zilliz Cloud 提供了全托管服务。",
    "BGE 是一个强大的中文向量模型。",
    "Python 是一种广泛用于 AI 的语言。",
    "今天天气真不错。"
]

print("⚗️  正在生成向量...")
vectors = embeddings.embed_documents(documents)
dim = len(vectors[0])
print(f"✅ 向量维度: {dim}")

# ==========================================
# 3. 连接与建表 (使用 MilvusClient)
# ==========================================
print(f"🔌 正在连接 Zilliz Cloud...")
# ✅ 修复点 1: 建立连接 (直接实例化 Client，不需要 connections.connect)
client = MilvusClient(uri=ZILLIZ_URI, token=ZILLIZ_TOKEN)

COLLECTION_NAME = "bge_test_new_api"

# ✅ 修复点 2: 删除旧表 (Client 直接调用，不需要 utility)
if client.has_collection(COLLECTION_NAME):
    client.drop_collection(COLLECTION_NAME)

# ✅ 修复点 3: 创建表结构 (Create Schema)
# 新版 API 定义 Schema 更直观
schema = client.create_schema(
    auto_id=True, 
    enable_dynamic_field=False, 
    description="测试新版API"
)
schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)
schema.add_field(field_name="text", datatype=DataType.VARCHAR, max_length=1000)
schema.add_field(field_name="vector", datatype=DataType.FLOAT_VECTOR, dim=dim)

# ✅ 修复点 4: 创建索引参数 (Index Params)
index_params = client.prepare_index_params()
index_params.add_index(
    field_name="vector", 
    metric_type="COSINE", 
    index_type="AUTOINDEX"
)

# 正式建表 (一步到位：建表 + 建索引)
client.create_collection(
    collection_name=COLLECTION_NAME,
    schema=schema,
    index_params=index_params
)
print(f"📦 集合 {COLLECTION_NAME} 创建完成")

# ==========================================
# 4. 数据整理与插入 (Insert)
# ==========================================
# ✅ 修复点 5: 数据格式转换
# MilvusClient 推荐使用 "列表嵌套字典" 的格式 (Rows)，而不是原来的按列存储
data_rows = []
for doc, vec in zip(documents, vectors):
    data_rows.append({
        "text": doc,
        "vector": vec
    })

res = client.insert(collection_name=COLLECTION_NAME, data=data_rows)
print(f"📥 成功插入 {res['insert_count']} 条数据")

# ==========================================
# 5. 搜索 (Search)
# ==========================================
query_text = "什么数据库是云原生的？"
print(f"\n🔎 提问: {query_text}")
query_vector = embeddings.embed_query(query_text)

# ✅ 修复点 6: 执行搜索 (不再返回 Future/Coroutine，直接返回 list)
results = client.search(
    collection_name=COLLECTION_NAME,
    data=[query_vector],
    limit=2,
    output_fields=["text"], # 必填，否则只返回 ID
    search_params={"metric_type": "COSINE", "params": {}}
)

for hits in results:
    for hit in hits:
        print("-" * 30)
        # client.search 返回的结果直接就是字典，不需要 .entity.get()
        print(f"匹配内容: {hit['entity']['text']}")
        print(f"相似度:   {hit['distance']:.4f}")

🧠 正在加载模型: D:/ollama/bge-base-zh-v1.5 ...
⚗️  正在生成向量...
✅ 向量维度: 768
🔌 正在连接 Zilliz Cloud...
📦 集合 bge_test_new_api 创建完成
📥 成功插入 5 条数据

🔎 提问: 什么数据库是云原生的？
------------------------------
匹配内容: Milvus 是一个云原生的开源向量数据库。
相似度:   0.6850
------------------------------
匹配内容: Zilliz Cloud 提供了全托管服务。
相似度:   0.4414


## 查询数据库

In [9]:
import os
from dotenv import load_dotenv
from pymilvus import MilvusClient

# 1. 连接数据库
# 1. 尝试从环境变量获取 Token
# 第二个参数是默认值，如果没找到环境变量会报错或使用空字符串，建议做个判断
token = os.getenv("ZILLIZ_TOKEN")

if not token:
    raise ValueError("❌ 未找到环境变量 'ZILLIZ_TOKEN'，请检查配置！")

client = MilvusClient(
    uri="https://in03-f410f8aefc6d1ee.serverless.gcp-us-west1.cloud.zilliz.com",
    token = token
)

COLLECTION_NAME = "demo_text_search"

# 2. 查看当前集合里总共有多少条数据
# ⚠️ 注意: 某些旧版本 Zilliz 可能不支持 count(*)，如果报错请跳过这一步
try:
    res = client.query(
        collection_name=COLLECTION_NAME,
        filter="",  # 空过滤条件匹配所有
        output_fields=["count(*)"]
    )
    print(f"📊 当前数据总量: {res[0]['count(*)']} 条")
except Exception:
    print("📊 (跳过统计总量)")

# 3. 查询具体的文本内容 (类似 SELECT * FROM table LIMIT 5)
print("\n📋 正在查询前 5 条数据...")

results = client.query(
    collection_name=COLLECTION_NAME,
    # filter 是必填的，"id >= 0" 是最常用的“查所有”的写法 (假设 id 是 Int64)
    filter="id >= 0", 
    output_fields=["id", "text"], # ⚠️ 这里只看 id 和 text，不看 vector，因为向量太长了打印出来会刷屏
    limit=5
)

for item in results:
    print("-" * 30)
    print(f"ID:   {item['id']}")
    print(f"文本: {item['text']}")

📊 当前数据总量: 5 条

📋 正在查询前 5 条数据...
------------------------------
ID:   463454660883580265
文本: Milvus 是一个云原生的开源向量数据库。
------------------------------
ID:   463454660883580266
文本: Zilliz Cloud 提供了全托管服务。
------------------------------
ID:   463454660883580267
文本: BGE 是一个强大的中文向量模型。
------------------------------
ID:   463454660883580268
文本: Python 是一种广泛用于 AI 的语言。
------------------------------
ID:   463454660883580269
文本: 今天天气真不错。
